In [1]:
import argparse
import os
import pickle
import time

from importlib import metadata
import torch
try:
    try:
        if metadata.version("rsl-rl"):
            raise ImportError
    except metadata.PackageNotFoundError:
        if metadata.version("rsl-rl-lib") != "3.1.1":  #2.2.4
            raise ImportError
except (metadata.PackageNotFoundError, ImportError) as e:
    raise ImportError("Please uninstall 'rsl_rl' and install 'rsl-rl-lib==2.2.4'.") from e
from rsl_rl.runners import OnPolicyRunner

In [2]:
from bp000_env_cnoid import BP000Env as RLEnv

In [3]:
# 任意設定項目
# exp_name = 'friction-walking-terrain2-kp2000kd50-kpkdrand-ridho-model'  # ckpt = 4000
exp_name = 'friction-walking-terrain2-kp2000kd50-kpkdrand-15'
# exp_name = 'friction-walking-terrain2-kp2000kd50-kpkdrand-9'  # min_ankle_height 弊害
# exp_name = 'friction-walking-terrain2-kp2000kd50-kpkdrand-8'  #  暫定１位
# exp_name = 'friction-walking-terrain2-kp2000kd50-kpkdrand-6'
# exp_name = 'friction-walking-terrain2-kp2000kd50-kpkdrand-5'
ckpt = 100

action_scale = 1.0 # 動作のスケールを調整

In [4]:
# 既存のセルを置き換え
import pandas as pd
import numpy as np

# データ収集用のリスト
# action_data = []
obs_data = []
torque_data = []
step_data = []

# CSVファイルの準備
csv_filename = f'obs_data/{exp_name}_step_data.csv'
os.makedirs('obs_data', exist_ok=True)

In [5]:
def _obs_vec(obs):
    # TensorDict or dict → 'policy' を優先
    if isinstance(obs, dict) or hasattr(obs, "get"):
        if "policy" in obs:
            obs = obs["policy"]
    if torch.is_tensor(obs):
        return obs.detach().cpu().numpy().ravel()
    return np.asarray(obs, dtype=np.float32).ravel()

In [6]:
## set robot path fix collisiton 
ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))  # /userdir
robot_path = os.path.join(ROOT, "userdir", "humanoid_research_k", "robots", "kawada_base.simple_collision.urdf")

In [7]:
log_dir = f"logs/{exp_name}"
env_cfg, obs_cfg, reward_cfg, command_cfg, train_cfg = pickle.load(open(f"logs/{exp_name}/cfgs.pkl", "rb"))
reward_cfg["reward_scales"] = {}

In [8]:
## override
env_cfg["episode_length_s"] = 60.0
# command_cfg["lin_vel_x_range"] = [0.5, 0.5]
env_cfg['dt'] = 0.01
env_cfg['substeps'] = 10
# env_cfg["kd"] = 50
env_cfg['base_roll_noise'] = [0,0]
env_cfg['base_pitch_noise'] = [0,0]
env_cfg['termination_if_roll_greater_than'] = 150
env_cfg['termination_if_pitch_greater_than'] = 150
env_cfg['rotorInertia'] = 0.1
env_cfg["base_init_pos"] = [0.0, 0.0, 0.64]

In [9]:
reward_cfg

{'tracking_sigma': 0.25,
 'base_height_target': 0.64,
 'feet_height_target': 0.075,
 'reward_scales': {}}

In [10]:
env_cfg



{'num_actions': 12,
 'default_joint_angles': {'R_HIP_Y': 0.0,
  'R_HIP_R': 0.0,
  'R_HIP_P': -0.8,
  'R_KNEE': 1.6,
  'R_ANKLE_P': -0.8,
  'R_ANKLE_R': 0.0,
  'L_HIP_Y': 0.0,
  'L_HIP_R': 0.0,
  'L_HIP_P': -0.8,
  'L_KNEE': 1.6,
  'L_ANKLE_P': -0.8,
  'L_ANKLE_R': 0.0},
 'joint_names': ['R_HIP_Y',
  'R_HIP_R',
  'R_HIP_P',
  'R_KNEE',
  'R_ANKLE_P',
  'R_ANKLE_R',
  'L_HIP_Y',
  'L_HIP_R',
  'L_HIP_P',
  'L_KNEE',
  'L_ANKLE_P',
  'L_ANKLE_R'],
 'kp': 2000.0,
 'kd': 50.0,
 'termination_if_roll_greater_than': 150,
 'termination_if_pitch_greater_than': 150,
 'base_init_pos': [0.0, 0.0, 0.64],
 'base_init_quat': [1.0, 0.0, 0.0, 0.0],
 'episode_length_s': 60.0,
 'resampling_time_s': 4.0,
 'action_scale': 1.0,
 'simulate_action_latency': True,
 'clip_actions': 100.0,
 'dt': 0.01,
 'substeps': 10,
 'rotorInertia': 0.1,
 'base_roll_noise': [0, 0],
 'base_pitch_noise': [0, 0],
 'domain_rand': {'friction': [0.4, 1.1],
  'restitution': [0.0, 0.2],
  'kp': [1800.0, 2200.0],
  'kd': [25.0, 75.0]}}

In [11]:
env = RLEnv(
    num_envs=1,
    env_cfg=env_cfg,
    obs_cfg=obs_cfg,
    reward_cfg=reward_cfg,
    command_cfg=command_cfg,
    dt=env_cfg['dt'],
    substeps=env_cfg['substeps'],
    show_viewer=True,
    robot_urdf_path=robot_path,
)

In [12]:
runner = OnPolicyRunner(env, train_cfg, log_dir, device='cuda')
resume_path = os.path.join(log_dir, f"model_{ckpt}.pt")
runner.load(resume_path)
policy = runner.get_inference_policy(device='cuda')

obs, _ = env.reset()
cnt = 0

torques = env.sim.sbody.getTorques()

print("obs : ", obs["policy"])

# データを記録
step_data.append(cnt)
obs_data.append(_obs_vec(obs))
torque_data.append(torques.copy())

cnt += 1

--------------------------------------------------------------------------------
Resolved observation sets: 
	 policy :  ['policy']
	 critic :  ['policy']
--------------------------------------------------------------------------------
Actor MLP: MLP(
  (0): Linear(in_features=45, out_features=512, bias=True)
  (1): ELU(alpha=1.0)
  (2): Linear(in_features=512, out_features=256, bias=True)
  (3): ELU(alpha=1.0)
  (4): Linear(in_features=256, out_features=128, bias=True)
  (5): ELU(alpha=1.0)
  (6): Linear(in_features=128, out_features=12, bias=True)
)
Critic MLP: MLP(
  (0): Linear(in_features=45, out_features=512, bias=True)
  (1): ELU(alpha=1.0)
  (2): Linear(in_features=512, out_features=256, bias=True)
  (3): ELU(alpha=1.0)
  (4): Linear(in_features=256, out_features=128, bias=True)
  (5): ELU(alpha=1.0)
  (6): Linear(in_features=128, out_features=1, bias=True)
)
obs :  tensor([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        

In [13]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 1
Original actions :  tensor([[-0.0757, -0.2572, -0.1207,  0.4088, -1.2104, -0.1513,  0.1377,  0.1743,
          0.1504,  0.0387, -0.9382, -0.1902]], device='cuda:0')
Scaled actions :  tensor([[-0.0757, -0.2572, -0.1207,  0.4088, -1.2104, -0.1513,  0.1377,  0.1743,
          0.1504,  0.0387, -0.9382, -0.1902]], device='cuda:0')


/userdir/irsl_rl/rl_env_base.py:110: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.exact_actions = torch.tensor(actions, device=self.device, dtype=torch.float32) ## copy
/userdir/irsl_rl/rl_env_cnoid.py:103: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:254.)
  self.dof_pos = torch.tensor([self.convAnglesToGenesis(sbody.angleVector())]).to(torch.float32).to(self.device)


obs :  tensor([[-4.2367e-06, -7.9481e-03, -1.6194e-06,  5.0845e-10,  3.7698e-20,
         -1.0000e+00,  1.0000e+00,  0.0000e+00,  0.0000e+00, -8.5621e-08,
         -4.7636e-08, -1.4180e-04,  3.6466e-04, -1.9103e-04,  7.6633e-07,
          1.4362e-08,  3.0845e-08, -1.4168e-04,  3.6454e-04, -1.9103e-04,
         -3.4389e-08, -4.2811e-06, -2.3818e-06, -7.0895e-03,  1.8232e-02,
         -9.5524e-03,  3.8316e-05,  7.1809e-07,  1.5422e-06, -7.0859e-03,
          1.8227e-02, -9.5517e-03, -1.7194e-06, -7.5722e-02, -2.5717e-01,
         -1.2066e-01,  4.0877e-01, -1.2104e+00, -1.5132e-01,  1.3771e-01,
          1.7430e-01,  1.5035e-01,  3.8695e-02, -9.3823e-01, -1.9018e-01]],
       device='cuda:0')
torques: [-1.58788519e-16 -1.00915103e-15  2.37314235e-06  7.56007923e-06
  1.59905156e-06  6.12612605e-17 -6.47408406e-18 -7.17891594e-16
  2.37314235e-06  7.56007923e-06  1.59905156e-06 -8.78307676e-17]
データ収集: step 2


In [14]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 2
Original actions :  tensor([[ 0.0066, -0.2897, -0.0950,  0.2195, -1.6829, -0.3367,  0.1596,  0.7778,
          0.1670,  0.2062, -1.3953, -0.0567]], device='cuda:0')
Scaled actions :  tensor([[ 0.0066, -0.2897, -0.0950,  0.2195, -1.6829, -0.3367,  0.1596,  0.7778,
          0.1670,  0.2062, -1.3953, -0.0567]], device='cuda:0')
obs :  tensor([[ 0.1343, -0.1791, -0.3294, -0.0034, -0.0029, -1.0000,  1.0000,  0.0000,
          0.0000, -0.0159, -0.0074, -0.0177,  0.0394, -0.1077, -0.0556,  0.0322,
          0.0051,  0.0037,  0.0227, -0.0990, -0.0478, -0.0762, -0.0804, -0.1272,
          0.3187, -0.9732, -0.2805,  0.2341,  0.0274,  0.0791,  0.1384, -0.8853,
         -0.2065,  0.0066, -0.2897, -0.0950,  0.2195, -1.6829, -0.3367,  0.1596,
          0.7778,  0.1670,  0.2062, -1.3953, -0.0567]], device='cuda:0')
torques: [   8.50938274  200.          200.         -101.55543155 -200.
  200.           84.57617402 -200.           53.01157146  173.74282094
 -200.          157.79034172]
データ収集:

In [15]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 3
Original actions :  tensor([[-0.0096, -0.1727,  0.6500, -0.4778,  0.0304, -0.3113, -0.2730,  0.0913,
         -0.6736, -0.5571, -0.4641,  0.3739]], device='cuda:0')
Scaled actions :  tensor([[-0.0096, -0.1727,  0.6500, -0.4778,  0.0304, -0.3113, -0.2730,  0.0913,
         -0.6736, -0.5571, -0.4641,  0.3739]], device='cuda:0')
obs :  tensor([[ 0.1326, -0.4019, -0.4872, -0.0159, -0.0085, -0.9998,  1.0000,  0.0000,
          0.0000, -0.0119, -0.0318, -0.0380,  0.1072, -0.4086, -0.1426,  0.0850,
          0.0191,  0.0320,  0.0599, -0.3844, -0.0615,  0.0524, -0.1580, -0.0622,
          0.3101, -1.9384, -0.4180,  0.1958,  0.1107,  0.1848,  0.2100, -1.8681,
          0.0034, -0.0096, -0.1727,  0.6500, -0.4778,  0.0304, -0.3113, -0.2730,
          0.0913, -0.6736, -0.5571, -0.4641,  0.3739]], device='cuda:0')
torques: [ -55.75681384  200.           99.31873745   93.46085884 -200.
    7.72059435  -12.093483   -200.          -48.27189429  -81.60341867
 -200.           29.11934302]
データ収集:

In [16]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 4
Original actions :  tensor([[-0.2272,  0.5131,  0.4905, -0.4781,  1.0812, -0.5436, -0.0883, -0.2989,
         -0.7418, -0.3296,  0.2425,  0.2052]], device='cuda:0')
Scaled actions :  tensor([[-0.2272,  0.5131,  0.4905, -0.4781,  1.0812, -0.5436, -0.0883, -0.2989,
         -0.7418, -0.3296,  0.2425,  0.2052]], device='cuda:0')
obs :  tensor([[ 0.1385, -0.3394,  0.1383, -0.0305, -0.0145, -0.9994,  1.0000,  0.0000,
          0.0000, -0.0101, -0.0694, -0.0294,  0.1332, -0.6919, -0.2020,  0.0648,
          0.0402,  0.0657,  0.0826, -0.6496,  0.0357, -0.0056, -0.1993,  0.1302,
         -0.0151, -0.9905, -0.2392, -0.3429,  0.1022,  0.1554,  0.0390, -0.8829,
          0.7135, -0.2272,  0.5131,  0.4905, -0.4781,  1.0812, -0.5436, -0.0883,
         -0.2989, -0.7418, -0.3296,  0.2425,  0.2052]], device='cuda:0')
torques: [-200.            4.10389931 -200.         -200.          200.
  -19.01268398    5.25564181  -15.56716175  200.         -200.
  200.           21.44160332]
データ収集: step 5


In [17]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 5
Original actions :  tensor([[-0.3243,  0.3187, -0.6588, -0.2357,  0.0673,  0.2404,  0.5771, -0.6060,
         -0.4219,  0.1403, -0.1342, -0.5352]], device='cuda:0')
Scaled actions :  tensor([[-0.3243,  0.3187, -0.6588, -0.2357,  0.0673,  0.2404,  0.5771, -0.6060,
         -0.4219,  0.1403, -0.1342, -0.5352]], device='cuda:0')
obs :  tensor([[ 0.0445, -0.3543,  0.5138, -0.0432, -0.0182, -0.9989,  1.0000,  0.0000,
          0.0000, -0.0614, -0.0948,  0.0105,  0.1011, -0.7900, -0.3036,  0.0062,
          0.0546,  0.0920,  0.0736, -0.7183,  0.1058, -0.4107, -0.0030,  0.1484,
         -0.0051, -0.2248, -0.2878, -0.2355,  0.0592,  0.1338, -0.1167,  0.0970,
          0.2251, -0.3243,  0.3187, -0.6588, -0.2357,  0.0673,  0.2404,  0.5771,
         -0.6060, -0.4219,  0.1403, -0.1342, -0.5352]], device='cuda:0')
torques: [  51.43997406 -200.         -200.         -200.          200.
  -28.40990399   -2.11226844  200.          200.         -200.
  200.           48.64755123]
データ収集: step 6


In [18]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 6
Original actions :  tensor([[ 0.8664, -0.2477, -0.3843, -0.5470, -0.3741,  0.0333,  0.3742,  0.3007,
         -0.0555,  0.6685, -0.9721,  0.2429]], device='cuda:0')
Scaled actions :  tensor([[ 0.8664, -0.2477, -0.3843, -0.5470, -0.3741,  0.0333,  0.3742,  0.3007,
         -0.0555,  0.6685, -0.9721,  0.2429]], device='cuda:0')
obs :  tensor([[-0.0220,  0.2762,  0.2923, -0.0444, -0.0176, -0.9989,  1.0000,  0.0000,
          0.0000, -0.1541, -0.0943,  0.0060,  0.1058, -0.7416, -0.2653,  0.0222,
          0.0571,  0.0730,  0.0885, -0.5971,  0.0268, -0.5227,  0.0209, -0.1675,
          0.0364,  0.6310,  0.5877,  0.3114, -0.0041, -0.2446,  0.1224,  1.0021,
         -0.8184,  0.8664, -0.2477, -0.3843, -0.5470, -0.3741,  0.0333,  0.3742,
          0.3007, -0.0555,  0.6685, -0.9721,  0.2429]], device='cuda:0')
torques: [ 200.         -200.         -200.            3.99992429  142.52002772
 -200.         -200.          200.         -200.         -200.
  200.          200.        ]
データ収集:

In [19]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 7
Original actions :  tensor([[ 0.8395, -0.8981,  0.7597, -0.2698, -0.7102, -0.2795, -0.6260,  0.4064,
          1.1878, -0.4782, -0.8432,  0.3723]], device='cuda:0')
Scaled actions :  tensor([[ 0.8395, -0.8981,  0.7597, -0.2698, -0.7102, -0.2795, -0.6260,  0.4064,
          1.1878, -0.4782, -0.8432,  0.3723]], device='cuda:0')
obs :  tensor([[-0.1011,  0.4170, -0.3327, -0.0283, -0.0147, -0.9995,  1.0000,  0.0000,
          0.0000, -0.1954, -0.0973, -0.0397,  0.1104, -0.6064, -0.1377,  0.1288,
          0.0611,  0.0103,  0.1340, -0.5024, -0.0305,  0.0358, -0.0456, -0.2497,
          0.0141,  0.5930,  0.5432,  0.5738,  0.0337, -0.3177,  0.2942,  0.0362,
          0.1445,  0.8395, -0.8981,  0.7597, -0.2698, -0.7102, -0.2795, -0.6260,
          0.4064,  1.1878, -0.4782, -0.8432,  0.3723]], device='cuda:0')
torques: [ -52.16307767  200.          200.          200.         -200.
  200.          200.         -200.         -196.94576781 -200.
 -195.03576765 -200.        ]
データ収集: step 8


In [20]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 8
Original actions :  tensor([[-0.1425,  0.9852,  0.3565, -0.2506, -0.5139,  0.0552, -0.4852,  0.0475,
          0.3431, -0.8022, -0.7066, -0.4382]], device='cuda:0')
Scaled actions :  tensor([[-0.1425,  0.9852,  0.3565, -0.2506, -0.5139,  0.0552, -0.4852,  0.0475,
          0.3431, -0.8022, -0.7066, -0.4382]], device='cuda:0')
obs :  tensor([[-0.1465, -0.3127, -0.3346, -0.0275, -0.0098, -0.9996,  1.0000,  0.0000,
          0.0000, -0.1357, -0.1102, -0.0467,  0.1092, -0.5886, -0.1025,  0.1879,
          0.0801, -0.0045,  0.1585, -0.5850,  0.0778,  0.5134, -0.0780,  0.1367,
         -0.0173, -0.2314, -0.1204,  0.0641,  0.1566,  0.1249, -0.0179, -0.5586,
          0.6294, -0.1425,  0.9852,  0.3565, -0.2506, -0.5139,  0.0552, -0.4852,
          0.0475,  0.3431, -0.8022, -0.7066, -0.4382]], device='cuda:0')
torques: [-200.          200.          200.         -200.           35.16034897
  -36.26767366  200.         -200.          200.         -200.
  -41.64527877 -200.        ]
データ収集:

In [21]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 9
Original actions :  tensor([[-0.7224, -0.3227, -0.8093, -0.0080, -0.0077,  0.4768, -0.2119, -1.4527,
         -1.0729,  0.2033, -0.5849, -0.6606]], device='cuda:0')
Scaled actions :  tensor([[-0.7224, -0.3227, -0.8093, -0.0080, -0.0077,  0.4768, -0.2119, -1.4527,
         -1.0729,  0.2033, -0.5849, -0.6606]], device='cuda:0')
obs :  tensor([[-1.4253e-01, -8.6309e-01,  3.3604e-01, -5.2106e-02, -4.0756e-03,
         -9.9863e-01,  1.0000e+00,  0.0000e+00,  0.0000e+00, -9.5622e-02,
         -1.1398e-01,  3.2510e-03,  9.3307e-02, -5.7809e-01, -5.8648e-02,
          1.3565e-01,  1.0859e-01,  6.1770e-02,  1.1964e-01, -6.5016e-01,
          9.7078e-02, -9.8202e-03, -7.3514e-04,  3.2950e-01, -1.1421e-01,
          9.9845e-02,  2.4158e-01, -5.3275e-01,  1.1793e-01,  5.0599e-01,
         -3.3888e-01, -2.7964e-01, -3.3913e-01, -7.2238e-01, -3.2266e-01,
         -8.0931e-01, -7.9670e-03, -7.6946e-03,  4.7680e-01, -2.1191e-01,
         -1.4527e+00, -1.0729e+00,  2.0330e-01, -5.8486e-01, -6.6

In [22]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 10
Original actions :  tensor([[ 0.1605,  0.5842, -0.6476,  0.1323, -0.5048, -0.4369,  1.3347,  0.7214,
         -1.3107, -0.2413, -0.2672,  0.4976]], device='cuda:0')
Scaled actions :  tensor([[ 0.1605,  0.5842, -0.6476,  0.1323, -0.5048, -0.4369,  1.3347,  0.7214,
         -1.3107, -0.2413, -0.2672,  0.4976]], device='cuda:0')
obs :  tensor([[ 0.4081, -0.2570,  0.6693, -0.0731, -0.0089, -0.9973,  1.0000,  0.0000,
          0.0000, -0.1593, -0.1399,  0.0319,  0.0733, -0.4556,  0.0812,  0.0142,
          0.0999,  0.1233,  0.0813, -0.6403, -0.0740, -0.5713, -0.2271, -0.0148,
         -0.0870,  0.9295,  0.8499, -0.5428, -0.1637,  0.1330, -0.0685,  0.1243,
         -1.1979,  0.1605,  0.5842, -0.6476,  0.1323, -0.5048, -0.4369,  1.3347,
          0.7214, -1.3107, -0.2413, -0.2672,  0.4976]], device='cuda:0')
torques: [ 200.         -200.         -200.          200.          -12.42162784
  -38.57369151 -200.         -165.26886524 -200.          -77.13028562
    9.70759952  -48.8167303

In [23]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 11
Original actions :  tensor([[ 0.7438, -0.0430,  0.3567, -0.1844, -1.3648, -0.6900,  0.5997,  0.4101,
          0.2940, -0.2247, -0.0892,  0.7901]], device='cuda:0')
Scaled actions :  tensor([[ 0.7438, -0.0430,  0.3567, -0.1844, -1.3648, -0.6900,  0.5997,  0.4101,
          0.2940, -0.2247, -0.0892,  0.7901]], device='cuda:0')
obs :  tensor([[-0.1055,  0.4474, -0.0291, -0.0684, -0.0130, -0.9976,  1.0000,  0.0000,
          0.0000, -0.2188, -0.1543, -0.0034,  0.0783, -0.3741,  0.1428, -0.0224,
          0.0929,  0.1207,  0.0571, -0.5372, -0.2070, -0.0759,  0.0579, -0.2930,
          0.0871, -0.0202, -0.1347,  0.1108,  0.0717, -0.1240, -0.1666,  0.5873,
         -0.2292,  0.7438, -0.0430,  0.3567, -0.1844, -1.3648, -0.6900,  0.5997,
          0.4101,  0.2940, -0.2247, -0.0892,  0.7901]], device='cuda:0')
torques: [ 200.          200.         -200.         -200.          -44.54825989
  200.          200.          200.         -200.           33.63889337
 -200.         -200.       

In [40]:
# 既存のforループを置き換え
num_steps = 100
for i in range(num_steps):
    with torch.no_grad():
        actions = policy(obs)
        
        # アクションスケーリング
        scaled_actions = actions * action_scale
        obs, rews, dones, infos = env.step(scaled_actions)  # スケール済みを使用
        torques = env.sim.sbody.getTorques()
        
        # データを記録
        step_data.append(cnt)
        obs_data.append(_obs_vec(obs))
        torque_data.append(torques.copy())
        
        # デバッグ表示（最初の数ステップのみ）
        if i < 3:
            print(f"Step {i}: Original action max={actions.max():.3f}, "
                  f"Scaled action max={scaled_actions.max():.3f}")
        
        if i % 20 == 0:
            print(f"Step {i+1}/{num_steps}, Total steps: {cnt}")
            print("steps:",cnt)
            print("actions :",scaled_actions)
            print("target_dof_pos:",env.target_dof_pos)
        
        cnt += 1

print(f"データ収集完了: {num_steps} steps collected with action_scale={action_scale}")

Step 0: Original action max=0.853, Scaled action max=0.853
Step 1/100, Total steps: 1412
steps: 1412
actions : tensor([[ 0.1865, -0.2319, -0.3043, -0.9494, -0.8549, -0.2448,  0.5157,  0.1486,
          0.8526, -1.8828,  0.6584, -0.1103]], device='cuda:0')
target_dof_pos: tensor([[-0.8264, -0.3011,  0.7780,  0.8741, -1.4339, -0.1106, -0.0710,  0.5499,
          0.4072,  0.5723, -0.5855,  0.7345]], device='cuda:0')
Step 1: Original action max=0.641, Scaled action max=0.641
Step 2: Original action max=0.826, Scaled action max=0.826
Step 21/100, Total steps: 1432
steps: 1432
actions : tensor([[ 0.2902,  0.3510, -0.7327,  0.2329, -1.2868,  0.8335, -1.2208, -0.8308,
         -0.3025,  0.1236,  0.5675, -0.3464]], device='cuda:0')
target_dof_pos: tensor([[-1.1499,  0.3834, -1.8312,  0.8299, -2.2549,  0.7793, -0.0857, -0.8548,
         -0.0367,  1.1321, -0.2574,  0.4064]], device='cuda:0')
Step 41/100, Total steps: 1452
steps: 1452
actions : tensor([[ 0.1488, -0.1376, -0.6849, -0.0314, -0.2152,

In [41]:
# for i in range(500):
#     with torch.no_grad():
#         actions = policy(obs)
#         scaled_actions = actions * action_scale
#         obs, rews, dones, infos = env.step(scaled_actions)

In [42]:
env.sim.stop()

In [37]:
env.reset()
cnt = 0

In [61]:
# 最もシンプルな保存方法
def save_simple_csv():
    if not step_data:
        print("データがありません")
        return
    
    # 基本的な辞書形式でデータを整理
    data_dict = {'step': step_data}
    
    # # Actionデータ
    # action_array = np.array(action_data)
    # for i in range(action_array.shape[1]):
    #     data_dict[f'action_{i}'] = action_array[:, i]
    
    # Observationデータ
    obs_array = np.array(obs_data)
    for i in range(obs_array.shape[1]):
        data_dict[f'obs_{i}'] = obs_array[:, i]
    
    # Torqueデータ
    torque_array = np.array(torque_data)
    for i in range(torque_array.shape[1]):
        data_dict[f'torque_{i}'] = torque_array[:, i]
    
    # DataFrameを作成して保存
    df = pd.DataFrame(data_dict)
    csv_filename = f'obs_data/cnoid_{exp_name}_ckpt{ckpt}_scale{action_scale}_rotorInertia0.1.csv'
    df.to_csv(csv_filename, index=False)
    
    print(f"シンプル版を保存: {csv_filename}")
    print(f"データ形状: {df.shape}")
    
    return df

# シンプル版を実行
df_simple = save_simple_csv()

シンプル版を保存: obs_data/cnoid_friction-walking-terrain2-kp2000kd50-kpkdrand-14_ckpt100_scale1.0_rotorInertia0.1.csv
データ形状: (362, 58)
